# Evaluating skill retrieval
What we want to do is the benchmark skill retrieval performance using the SkillRet dataset on Hugging Face.

# License
SkillRet use the Apache License Version 2.0, it allows us to use, reproduce, modify, and redistribute the dataset free of charge. The author of the dataset (Kang, Ryangkyung and Cho, Hongcheol and Kim, Youngeun) want us to cite their paper (SkillRet: A Large-Scale Benchmark for Skill Retrieval in LLM Agents) in our work.

# The two-day trial run
## Requirements for the trial run
1. Download the dataset
2. Run a simple baseline
3. Print one metric

## How we want to accomplish it
1. Download SkillRet
2. Inspect the files, format and size
3. Run our baseline (BM25) on a subset of the test data
4. Display the result of Recall@5, which is a metric that evaluate if the 'Gold skill' is found in the 5 retrived skill of the retrivel method benchmarked

In [ ]:
# Import required libraries
import pandas as pd
import bm25s
from datasets import load_dataset
from datasets import load_dataset_builder

In [ ]:
from datasets import load_dataset_builder

for name in ["skills", "queries", "qrels"]:
    info = load_dataset_builder("ThakiCloud/SKILLRET", name).info

    # Sizes in bytes, so divide by 1024^2 to get megabytes
    print(name)
    print("  Dataset size:", int(info.dataset_size / 1024**2), "MB")
    print()

In [ ]:
skills = load_dataset("ThakiCloud/SKILLRET", "skills", split="test")
queries = load_dataset("ThakiCloud/SKILLRET", "queries", split="test")
qrels = load_dataset("ThakiCloud/SKILLRET", "qrels", split="test")

In [ ]:
# Exploring the skills

# Number of skills/queries/qrel 
# Qrels gives the correct gold skill for the query 
print(f"Skills:  {len(skills):,}")
print(f"Queries: {len(queries):,}")
print(f"Qrels:   {len(qrels):,}")

# Using pandas give a nicer display of the dataset
display(skills.select(range(3)).to_pandas())
display(queries.select(range(3)).to_pandas())
display(qrels.select(range(3)).to_pandas())

In [ ]:
# Combine the text from each skill into one document so BM25 can search it
documents = [
    f"{x['name']} {x['description']} {x['skill_md']}"
    for x in skills
]

# Create the BM25 search engine
bm25 = bm25s.BM25()

# Split the skill documents into searchable tokens
tokens = bm25s.tokenize(documents)

# Build the BM25 index so we can search the skill library.
bm25.index(tokens)

In [ ]:
# Use 20 queries for the trial
test_queries = queries.select(range(20))

gold = {}

# Create the golden skill dictionary (query_id with the relevant skill(s) ID)
for row in qrels:
    if row["relevance"] > 0:
        gold.setdefault(row["query_id"], set()).add(row["skill_id"])

# Preview the dict
gold_preview = [
    {
        "query_id": query_id,
        "relevant_skill_ids": ", ".join(sorted(skill_ids))
    }
    for query_id, skill_ids in list(gold.items())[:5]
]

display(pd.DataFrame(gold_preview))


recalls = []

for query in test_queries:
    # Search the skill library with BM25.
    query_tokens = bm25s.tokenize(query["query"])
    results, scores = bm25.retrieve(query_tokens, k=5)

    # Convert BM25 result to SkillRet skill(s) ID
    predicted = {
        skills[int(i)]["id"]
        for i in results[0]
    }

    # Get the correct skill(s) for this query
    expected = gold[query["id"]]

    # Recall@5 = fraction of relevant skills found in the top 5
    recall = len(predicted & expected) / len(expected)
    recalls.append(recall)

In [ ]:
# What Recall@5 does is that it look if the golden skill is in the 5 top retrieved skill
# If there are multiple skills, the value can be fractional depending on the number of golden skills found
print(recalls)
print()

# Average Recall@5 across the 20 test queries
recall_at_5 = sum(recalls) / len(recalls)

print(f"BM25 Recall@5: {recall_at_5:.3f}")